In [1]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from skorch import NeuralNetRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
df= pd.read_csv('diamonds.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  str    
 2   color    53940 non-null  str    
 3   clarity  53940 non-null  str    
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   x        53940 non-null  float64
 7   y        53940 non-null  float64
 8   z        53940 non-null  float64
 9   price    53940 non-null  int64  
dtypes: float64(6), int64(1), str(3)
memory usage: 4.1 MB


In [4]:
df = pd.get_dummies(df, drop_first=True)
X = df.drop(columns=['price']).values
y = df['price'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler_X = StandardScaler()
X_train = scaler_X.fit_transform(X_train)
X_test = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

In [5]:
df

,carat,depth,table,x,y,z,price,cut_Good,cut_Ideal,cut_Premium,...,color_H,color_I,color_J,clarity_IF,clarity_SI1,clarity_SI2,clarity_VS1,clarity_VS2,clarity_VVS1,clarity_VVS2
0,0.23,61.5,55.0,3.95,3.98,2.43,326,False,True,False,...,False,False,False,False,False,True,False,False,False,False
1,0.21,59.8,61.0,3.89,3.84,2.31,326,False,False,True,...,False,False,False,False,True,False,False,False,False,False
2,0.23,56.9,65.0,4.05,4.07,2.31,327,True,False,False,...,False,False,False,False,False,False,True,False,False,False
3,0.29,62.4,58.0,4.20,4.23,2.63,334,False,False,True,...,False,True,False,False,False,False,False,True,False,False
4,0.31,63.3,58.0,4.34,4.35,2.75,335,True,False,False,...,False,False,True,False,False,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53935,0.72,60.8,57.0,5.75,5.76,3.50,2757,False,True,False,...,False,False,False,False,True,False,False,False,False,False
53936,0.72,63.1,55.0,5.69,5.75,3.61,2757,True,False,False,...,False,False,False,False,True,False,False,False,False,False
53937,0.70,62.8,60.0,5.66,5.68,3.56,2757,False,False,False,...,False,False,False,False,True,False,False,False,False,False
53938,0.86,61.0,58.0,6.15,6.12,3.74,2757,False,False,True,...,True,False,False,False,False,True,False,False,False,False


In [6]:
X_tr = torch.FloatTensor(X_train)
y_tr = torch.FloatTensor(y_train).reshape(-1, 1)
X_te = torch.FloatTensor(X_test)
y_te = torch.FloatTensor(y_test)

In [7]:
batch_size = 256
train_dataset = torch.utils.data.TensorDataset(X_tr, y_tr)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,
                                           batch_size=batch_size,
                                           shuffle=True)

In [8]:
class NeuralNet2(nn.Module):
    def __init__(self, input_size=23, hidden_size=64, num_classes=1):
        super(NeuralNet2, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

model = NeuralNet2()

num_epochs = 50
learning_rate = 0.01

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [12]:
net = NeuralNetRegressor(
    NeuralNet2,
    module__input_size=23,
    module__num_classes=1,
    criterion=nn.MSELoss,
    optimizer=torch.optim.Adam,
    max_epochs=50,
    lr=0.01,
    batch_size=256,
    verbose=0
)

param_grid = {
    'module__hidden_size': [32, 64, 128],
    'lr': [0.001, 0.01],
    'batch_size': [128, 256],
    'max_epochs': [50, 100]
}

In [13]:
grid = GridSearchCV(
    estimator=net,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=2,
    n_jobs=-1
)

In [14]:
grid.fit(
    X_tr,
    y_tr
)

print("\nЛучшие гиперпараметры:")
print(grid.best_params_)

print("\nЛучший score:")
print(grid.best_score_)

best_model = grid.best_estimator_

best_model.module_.eval()

Fitting 3 folds for each of 24 candidates, totalling 72 fits

Лучшие гиперпараметры:
{'batch_size': 128, 'lr': 0.001, 'max_epochs': 50, 'module__hidden_size': 32}

Лучший score:
-0.0389765507231156


NeuralNet2(
  (fc1): Linear(in_features=23, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)

In [16]:
with torch.no_grad():

    pred_scaled = best_model.predict(X_te)

    pred_scaled = np.array(pred_scaled).reshape(-1, 1)

pred_original = scaler_y.inverse_transform(
    pred_scaled
)

y_test_original = scaler_y.inverse_transform(
    y_te.numpy().reshape(-1, 1)
)

mae = mean_absolute_error(
    y_test_original,
    pred_original
)

mse = mean_squared_error(
    y_test_original,
    pred_original
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test_original,
    pred_original
)

print("\nМетрики лучшей модели:")

print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


Метрики лучшей модели:
MAE:  351.9217
MSE:  388108.6562
RMSE: 622.9837
R²:   0.9756
